# Terra-Style Jupyter Demo

This notebook is part of the Heartwood open-source project. Copyright and license metadata are declared in `REUSE.toml`.

This notebook walks through the synthetic Heartwood demo path for a Terra-like Jupyter workspace. It uses the same session command/event contract as the CLI and researcher web UI, keeps all data synthetic, and does not claim real Terra identity binding or controlled-data validation.


## Terminal Setup

Use the `edge-smoke` image when the demo must include the bundled tiny local model artifact and the full offline smoke path. From a Jupyter terminal inside that image, run:

```bash
bash images/generic/scripts/offline_stack_smoke.sh
```

To launch the packaged researcher web UI from the same workspace, start the gateway in a Jupyter terminal:

```bash
export HEARTWOOD_WORKSPACE=/home/jupyter/heartwood-workspace/sessions
export HEARTWOOD_WEB_HOST=127.0.0.1
export HEARTWOOD_WEB_PORT=8767
bash images/generic/scripts/start_web_ui.sh
```

Open the notebook proxy URL for port `8767`. In Terra-style `jupyter-server-proxy` routes, the browser usually sees `/user/<name>/proxy/8767/` while the proxy strips that prefix before forwarding to Heartwood.


In [ ]:
from pathlib import Path

from heartwood.notebook import NotebookSession, build_widget_spec, jupyter_proxy_url

workspace = Path.home() / "heartwood-workspace" / "sessions"
session = NotebookSession(workspace=workspace, session_id="terra-demo")

print("Web UI proxy URL:", jupyter_proxy_url(port=8767))
print("Session workspace:", workspace)


In [ ]:
detection = session.detect()

print("Dataset proposals")
for proposal in detection.dataset_proposals:
    print(f"- {proposal.dataset_type} ({proposal.confidence:.2f})")
    for item in proposal.evidence:
        print(f"  - {item}")

print("Latest activity:", detection.activity[-1])


In [ ]:
run = session.run("run the synthetic workflow")

print("Policy decisions")
for policy in run.policy_status:
    print(f"- {policy.decision}: {policy.endpoint} ({policy.reason})")

print("Approval controls")
for control in run.approval_controls:
    print(f"- {control.target_type}: {control.target_id} [{control.decision or 'pending'}]")

print("Activity")
for item in run.activity:
    print(f"{item.sequence:03d} {item.kind}: {item.detail}")


In [ ]:
exported = session.audit_export()
replayed = session.replay()

print("Persisted events:", replayed.event_count)
print("Exports")
for action in exported.export_actions:
    print(f"- {action.label}: {action.path}")

print("Notebook widget sections:", [section.title for section in build_widget_spec(replayed)])


## CLI Replay Evidence

The CLI reads the same persisted session events. From a Jupyter terminal, use:

```bash
heartwood --workspace /home/jupyter/heartwood-workspace/sessions --session-id terra-demo replay
```

A complete synthetic demo should show the offline smoke output, this notebook activity, the proxied web UI loaded through the notebook route, CLI replay for the same session, a scrubbed audit export, and a synthetic reviewer packet. A live Terra validation pass still needs to confirm image launch, Leonardo proxy behavior, inherited identity, and platform-specific data/model controls.
